In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
import os
import random
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, TimesformerForVideoClassification
import torch
from video_processing import VideoDataset

c:\Users\parth\.pyenv-win-venv\envs\capdis_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_label(path):
  return path.split('/')[2]

In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TimesformerForVideoClassification.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    trust_remote_code=True,
    use_safetensors=True,
    ignore_mismatched_sizes=True,
    num_labels=10,
)
model.eval().to(device)

Some weights of TimesformerForVideoClassification were not initialized from the model checkpoint at facebook/timesformer-base-finetuned-k400 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TimesformerForVideoClassification(
  (timesformer): TimesformerModel(
    (embeddings): TimesformerEmbeddings(
      (patch_embeddings): TimesformerPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (time_drop): Dropout(p=0.0, inplace=False)
    )
    (encoder): TimesformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x TimesformerLayer(
          (drop_path): Identity()
          (attention): TimeSformerAttention(
            (attention): TimesformerSelfAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
            )
            (output): TimesformerSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): TimesformerIntermediate(
            (dense

In [4]:
video_folder = "data/"
data = {}
video_paths,labels = [],[]
for root, _, files in os.walk(video_folder):
    for f in files:
        if f.endswith((".mp4", ".avi")):
            path=os.path.join(root, f).replace("\\", "/")
            data[path] = extract_label(path)
            video_paths.append(path)
            labels.append(extract_label(path))


In [5]:
# from collections import defaultdict
# label_to_paths = defaultdict(list)

# for path, label in data.items():
#     label_to_paths[label].append(path)
# balanced_sample = {}
# for label, paths in label_to_paths.items():
#     num_samples = min(200, len(paths))
#     selected_paths = random.sample(paths, num_samples)
#     for path in selected_paths:
#         balanced_sample[path] = label

In [6]:
# idxs = [random.randint(1, len(labels)) for i in range(100)]
# video_paths, labels = [video_paths[x] for x in idxs], [labels[x] for x in idxs]
# len(video_paths), len(labels)

In [7]:
# video_paths, labels = list(balanced_sample.keys()), list(balanced_sample.values())

In [8]:
class_names = set(labels)
class_names

{'corner',
 'foul',
 'freekick',
 'goalkick',
 'longpass',
 'ontarget',
 'penalty',
 'shortpass',
 'substitution',
 'throw-in'}

In [9]:
id2label = {i: label for i, label in enumerate(class_names)}
label2id = {label: i for i, label in enumerate(class_names)}
model.config.num_labels = 10
model.config.id2label = id2label
model.config.label2id = label2id

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    video_paths, labels, test_size=0.2, random_state=42, stratify=labels
)
len(X_train), len(X_val), len(y_train), len(y_val)

(2888, 722, 2888, 722)

In [11]:
# X_train, X_val, y_train, y_val = video_paths[5:], video_paths[:5], labels[5:], labels[:5]
# len(X_train), len(X_val), len(y_train), len(y_val)

In [12]:
for label in set(labels):
    print(f"{label}: {labels.count(label)}")

ontarget: 340
freekick: 374
substitution: 336
penalty: 322
corner: 411
throw-in: 367
goalkick: 425
shortpass: 354
longpass: 381
foul: 300


In [13]:
y_train_encoded, uniques = pd.factorize(y_train)
y_val_encoded = pd.Series(y_val).map({label: idx for idx, label in enumerate(uniques)}).values

C:\Users\parth\AppData\Local\Temp\ipykernel_21556\4134361369.py:1: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  y_train_encoded, uniques = pd.factorize(y_train)


In [14]:
train_dataset = VideoDataset(X_train, y_train_encoded, num_frames=20)
val_dataset = VideoDataset(X_val, y_val_encoded, num_frames=20)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [16]:
# train_features, train_labels = next(iter(train_loader))
# print(f"Feature batch shape: {train_features.size()}")
# print(f"Labels batch shape: {train_labels.size()}")
# img = train_features[0].squeeze()
# label = train_labels[0]
# for frame in img:
#     plt.imshow(frame[0], cmap="gray")
#     plt.show()
# print(f"Label: {id2label[int(label)]}")

In [17]:
print(model.classifier)

Linear(in_features=768, out_features=10, bias=True)


Significant imporvement upon 2nd epoch, try 3-4 more

In [18]:
scaler = torch.amp.GradScaler("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
for epoch in range(5):
    model.train()
    epoch_loss = 0

    for batch_x, batch_y in tqdm(train_loader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(batch_x, labels=batch_y)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        del batch_x, batch_y, outputs, loss
        torch.cuda.empty_cache()
    print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(train_loader):.4f}")

100%|██████████| 2888/2888 [36:06<00:00,  1.33it/s]


Epoch 1, Loss: 1.1960


100%|██████████| 2888/2888 [36:02<00:00,  1.34it/s]


Epoch 2, Loss: 0.5618


100%|██████████| 2888/2888 [35:17<00:00,  1.36it/s]


Epoch 3, Loss: 0.3927


100%|██████████| 2888/2888 [36:27<00:00,  1.32it/s]


Epoch 4, Loss: 0.2897


100%|██████████| 2888/2888 [38:33<00:00,  1.25it/s]

Epoch 5, Loss: 0.2273


In [19]:
from sklearn.metrics import classification_report
import torch

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in tqdm(val_loader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x).logits
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
        del batch_x, batch_y, outputs, preds
        torch.cuda.empty_cache()

        

# Generate classification report
report = classification_report(all_labels, all_preds, digits=4)
print(report)


100%|██████████| 722/722 [08:17<00:00,  1.45it/s]

              precision    recall  f1-score   support

           0     0.9531    0.8133    0.8777        75
           1     0.7857    0.9041    0.8408        73
           2     0.9367    0.9737    0.9548        76
           3     0.8462    0.9851    0.9103        67
           4     0.9425    0.9647    0.9535        85
           5     1.0000    0.7167    0.8350        60
           6     0.9296    0.9706    0.9496        68
           7     0.9846    0.9014    0.9412        71
           8     0.9259    0.9146    0.9202        82
           9     0.8714    0.9385    0.9037        65

    accuracy                         0.9114       722
   macro avg     0.9176    0.9083    0.9087       722
weighted avg     0.9176    0.9114    0.9107       722



In [ ]:
model.save_pretrained('./_timesformer_finetuned_model')